In [11]:
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports successful.')

Imports successful.


In [12]:
# Load features
X = np.load('../data/X_features.npy')
y = np.load('../data/y_labels.npy')

# Encode labels to numbers
le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f'Classes: {le.classes_}')
print(f'Encoded: {np.unique(y_enc)}')

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Encoded: [0 1 2 3 4 5 6 7]


In [13]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')
print(f'Feature size: {X_train.shape[1]}')

Train size: 2304
Test size:  576
Feature size: 112


In [14]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

models = {
    'SVM_rbf_C10': SVC(kernel='rbf', C=10, gamma='scale'),
    'SVM_rbf_C1': SVC(kernel='rbf', C=1, gamma='scale'),
    'SVM_linear': SVC(kernel='linear', C=1),
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }

    print(f'Accuracy: {results[name]["accuracy"]:.4f}')
    print(f'F1 Score: {results[name]["f1"]:.4f}\n')

Training SVM_rbf_C10...
Accuracy: 0.9167
F1 Score: 0.9162

Training SVM_rbf_C1...
Accuracy: 0.8351
F1 Score: 0.8330

Training SVM_linear...
Accuracy: 0.8403
F1 Score: 0.8402



In [15]:
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)

print('Training SVM...')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')

Training SVM...
Accuracy:  0.9167
Precision: 0.9188
Recall:    0.9167
F1 Score:  0.9162


In [16]:
print('\nClassification Report:')
print(classification_report(y_test, y_pred, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.89      0.92        76
           1       0.87      1.00      0.93        77
           2       0.89      0.84      0.87        77
           3       0.92      0.95      0.94        77
           4       0.95      0.92      0.93        77
           5       1.00      0.84      0.91        38
           6       0.92      0.87      0.89        77
           7       0.90      0.97      0.94        77

    accuracy                           0.92       576
   macro avg       0.92      0.91      0.92       576
weighted avg       0.92      0.92      0.92       576



In [17]:
import numpy as np

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print(cm)


Confusion Matrix:
[[68  0  4  2  2  0  0  0]
 [ 0 77  0  0  0  0  0  0]
 [ 2  2 65  0  2  0  0  6]
 [ 0  2  0 73  0  0  2  0]
 [ 2  2  0  0 71  0  0  2]
 [ 0  2  0  0  0 32  4  0]
 [ 0  4  2  4  0  0 67  0]
 [ 0  0  2  0  0  0  0 75]]


In [18]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')

print('\nCross-validation F1 scores:', cv_scores)
print('Mean CV F1:', cv_scores.mean())


Cross-validation F1 scores: [0.90847653 0.94333535 0.89763185 0.911336   0.91326133]
Mean CV F1: 0.9148082130082399


In [19]:
import os
import joblib

# create directory
os.makedirs('../models', exist_ok=True)

# save SVM model
joblib.dump(model, '../models/svm_model_2.0.pkl')

# save preprocessing objects (if used)
joblib.dump(scaler, '../models/svm_scaler_2.0.pkl')
joblib.dump(le, '../models/svm_label_encoder_2.0.pkl')

print('SVM 2.0 model saved.')

SVM 2.0 model saved.
